# Intelligent PPE Monitoring - Colab GPU Test

Test the full YOLO + SAM 3 hybrid detection pipeline on a free T4 GPU.

## Setup
1. **Runtime > Change runtime type > T4 GPU**
2. Upload to Google Drive `MyDrive/ppe_models/`:
   - `backend_colab.zip` (from project root)
   - `best.pt` (YOLO model)
   - `sam3.pt` (SAM model)
3. Run all cells top-to-bottom

---
## 1. GPU Check

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')

In [ ]:
# from google.colab import drive
# import os
# from huggingface_hub import hf_hub_download

# # 1. Mount your Google Drive
# drive.mount('/content/drive')

# # 2. Define the exact target folder
# drive_folder = '/content/drive/MyDrive/ppe_models'
# os.makedirs(drive_folder, exist_ok=True)

# # 3. Insert your REAL token here (Make sure to replace the asterisks!)
# HF_TOKEN = "hugging face token"

# print("Starting download... (This may take a few minutes)")

# try:
#     # 4. Download directly using Hugging Face's official API
#     downloaded_path = hf_hub_download(
#         repo_id="facebook/sam3",
#         filename="sam3.pt",
#         local_dir=drive_folder,  # Saves directly to your Drive folder
#         token=HF_TOKEN
#     )
#     print(f"✅ Success! SAM 3 successfully downloaded to: {downloaded_path}")

# except Exception as e:
#     print("🚨 DOWNLOAD FAILED 🚨")
#     print(f"Error details: {e}")
#     print("\nHOW TO FIX THIS:")
#     print("1. Did you replace the asterisks in the token with your real token?")
#     print("2. Did you go to https://huggingface.co/facebook/sam3 and click 'Agree' to accept Meta's license?")

## 2. Install Dependencies (incl. SAM 3 CLIP)

In [ ]:
# Core dependencies
!pip install -q -U pillow ultralytics fastapi uvicorn[standard] python-multipart pyngrok \
    sqlalchemy pydantic>=2.0.0 pydantic-settings python-dotenv \
    opencv-python-headless Pillow reportlab apscheduler

# SAM 3 requires Ultralytics CLIP fork (NOT the pip 'clip' package)
!pip uninstall clip -y 2>/dev/null
!pip install -q git+https://github.com/ultralytics/CLIP.git
# Download Sam 3
# print("Downloading Sam 3")
# !wget --header="Authorization: Bearer hf token" "https://huggingface.co/facebook/sam3/resolve/main/sam3.pt"
# print("Sam 3 download completed")
import ultralytics
print(f'\nUltralytics: {ultralytics.__version__}')
print('All dependencies installed')

## 3. Mount Drive + Extract Backend + Copy Models

In [ ]:
from google.colab import drive
import os, shutil, zipfile

drive.mount('/content/drive')

# Configure paths (edit if yours are different)
DRIVE_DIR = '/content/drive/MyDrive/ppe_models'

# 1. Extract backend code
zip_path = f'{DRIVE_DIR}/backend_colab.zip'
assert os.path.exists(zip_path), f'{zip_path} not found! Upload backend_colab.zip to Drive.'

if os.path.exists('/content/backend'):
    shutil.rmtree('/content/backend')

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('/content')

assert os.path.exists('/content/backend/main.py'), 'main.py not found after extraction'
print('Backend code extracted')

# Verify __init__.py is NOT empty
init_path = '/content/backend/api/routes/__init__.py'
init_size = os.path.getsize(init_path)
assert init_size > 10, f'api/routes/__init__.py is empty ({init_size} bytes)!'
print(f'api/routes/__init__.py OK ({init_size} bytes)')

# 2. Create directories
for d in ['models', 'uploads', 'reports']:
    os.makedirs(f'/content/backend/{d}', exist_ok=True)

# 3. Copy models from Drive
for model_name in ['best.pt', 'sam3.pt']:
    src = f'{DRIVE_DIR}/{model_name}'
    dst = f'/content/backend/models/{model_name}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        mb = os.path.getsize(dst) / 1e6
        print(f'{model_name} copied ({mb:.1f} MB)')
    else:
        print(f'WARNING: {model_name} not found at {src}')

print('\nSetup complete!')

## 4. Write .env + Init Database

In [ ]:
import sys, os
os.chdir('/content/backend')
sys.path.insert(0, '/content/backend')

with open('.env', 'w') as f:
    f.write("""DATABASE_URL=sqlite:////content/backend/ppe_detection.db
YOLO_MODEL_PATH=/content/backend/models/best.pt
YOLO_CONFIDENCE_THRESHOLD=0.25
YOLO_IMGSZ=1280
SAM_ENABLED=true
SAM_MODEL_PATH=/content/backend/models/sam3.pt
SAM_DEVICE=cuda
SAM_MASK_THRESHOLD=0.05
VIOLATION_COOLDOWN_SECONDS=300
VIOLATION_IOU_THRESHOLD=0.3
VIOLATION_TRACK_TIMEOUT=60
DEFAULT_SITE_LOCATION=Colab Test Site
DEFAULT_CAMERA_ID=colab_cam_01
REPORT_OUTPUT_DIR=/content/backend/reports
REPORT_TIME=23:59
DEBUG=true
""")
print('.env written')

from database.connection import engine
from database.models import Base
Base.metadata.create_all(bind=engine)

from sqlalchemy import inspect
tables = inspect(engine).get_table_names()
print(f'Database ready - tables: {tables}')

## 5. Load Models + Smoke Test

In [ ]:
!pip install -U Pillow==10.0.0

In [ ]:
import time, numpy as np

# Fix for Pillow dependency conflict with ultralytics
# The error 'ImportError: cannot import name '_Ink' from 'PIL._typing''
# is often resolved by downgrading Pillow to a compatible version.
# !pip install -U Pillow==10.0.0

# Load YOLO (Sentry)
print('Loading YOLO (Sentry)...')
from services.yolo_detector import get_yolo_detector
yolo = get_yolo_detector()
yolo.load_model()
print(f'YOLO loaded - device: {yolo.device}')

# Load SAM 3 (Judge)
print('\nLoading SAM 3 (Judge)...')
from services.sam_verifier import get_sam_verifier
sam = get_sam_verifier()
sam.load_model()
print(f'SAM 3 loaded - mock mode: {sam.is_mock()}')

# Smoke test YOLO
blank = np.zeros((720, 1280, 3), dtype=np.uint8)
t0 = time.time()
r = yolo.detect(blank)
ms = (time.time() - t0) * 1000
print(f'\nYOLO smoke test: {ms:.0f}ms, {len(r["persons"])} persons')

## 6. Upload Image + Run Full Hybrid Detection

In [ ]:
from google.colab import files
import cv2, numpy as np

print('Upload a test image (JPG/PNG)...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
test_image = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv2.IMREAD_COLOR)
print(f'Loaded: {fname} ({test_image.shape[1]}x{test_image.shape[0]}px)')

In [ ]:
import time, os, sys
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

sys.path.insert(0, '/content/backend')
os.chdir('/content/backend')

from services.hybrid_detector import get_hybrid_detector
detector = get_hybrid_detector()

# Run detection
print('Running hybrid detection (YOLO + SAM 3)...')
t0 = time.time()
result = detector.detect(
    test_image,
    save_annotated=True,
    output_path='/content/backend/uploads/test_annotated.jpg'
)
total_ms = (time.time() - t0) * 1000

# Results
print(f'\nTotal: {total_ms:.0f}ms')
print(f'Persons: {result["stats"]["total_persons"]}')
print(f'Violations: {result["stats"]["total_violations"]}')
print(f'Compliance: {result["stats"]["compliance_rate"]:.1f}%')
print(f'SAM activations: {result["stats"]["sam_activations"]}')
print(f'SAM bypass rate: {result["stats"]["bypass_rate"]:.1f}%')

# Path distribution
print(f'\nPath distribution:')
for path, count in result['stats']['path_distribution'].items():
    print(f'  {path}: {count}')

# Per-person detail
for i, p in enumerate(result['persons']):
    v = 'VIOLATION' if p['is_violation'] else 'SAFE'
    print(f'  Person {i+1}: {v} Helmet:{"Y" if p["has_helmet"] else "N"} Vest:{"Y" if p["has_vest"] else "N"} Path:{p["decision_path"]} SAM:{p["sam_activated"]}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
img_rgb = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
axes[0].imshow(img_rgb)
axes[0].set_title('Original + Detections')
for p in result['persons']:
    x1, y1, x2, y2 = p['bbox']
    c = 'red' if p['is_violation'] else 'green'
    axes[0].add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, lw=2, ec=c, fc='none'))
    axes[0].text(x1, y1-5, p['decision_path'], color=c, fontsize=8, fontweight='bold')
axes[0].axis('off')

ann_path = '/content/backend/uploads/test_annotated.jpg'
if os.path.exists(ann_path):
    axes[1].imshow(cv2.cvtColor(cv2.imread(ann_path), cv2.COLOR_BGR2RGB))
    axes[1].set_title('Annotated Output')
else:
    axes[1].text(0.5, 0.5, 'No annotated image', ha='center')
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/content/detection_result.png', dpi=150)
plt.show()

## 7. FPS Benchmark (YOLO)

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt
from services.yolo_detector import get_yolo_detector
yolo = get_yolo_detector()

N = 20
latencies = []
for i in range(N):
    t0 = time.time()
    yolo.detect(test_image)
    latencies.append((time.time() - t0) * 1000)
    if (i+1) % 5 == 0: print(f'  {i+1}/{N}: {latencies[-1]:.1f}ms')

warm = latencies[3:]  # skip warmup
avg = np.mean(warm)
fps = 1000 / avg

print(f'\n{"="*40}')
print(f'YOLO BENCHMARK (Thesis)')
print(f'{"="*40}')
print(f'  Avg latency: {avg:.1f}ms')
print(f'  Min/Max: {np.min(warm):.1f} / {np.max(warm):.1f}ms')
print(f'  FPS: {fps:.1f}')
print(f'  Image size: 1280')
print(f'  GPU: {torch.cuda.get_device_name(0)}')

plt.figure(figsize=(10, 4))
plt.plot(warm, 'o-', color='#2196F3')
plt.axhline(avg, color='red', ls='--', label=f'Avg: {avg:.1f}ms ({fps:.1f} FPS)')
plt.xlabel('Run'); plt.ylabel('ms'); plt.title('YOLO Inference Latency')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/benchmark_yolo.png', dpi=150)
plt.show()

## 8. SAM 3 Verification Stats

In [ ]:
from services.sam_verifier import get_sam_verifier

sam = get_sam_verifier()
stats = sam.get_stats()

print('SAM 3 VERIFICATION STATS')
print('=' * 40)
for k, v in stats.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.2f}')
    else:
        print(f'  {k}: {v}')

## 9. Start API Server + ngrok

Get a free token at https://dashboard.ngrok.com/signup

In [ ]:
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # paste your token

import subprocess, threading, time, requests, os
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()

def run_server():
    env = os.environ.copy()
    env['PYTHONPATH'] = '/content/backend'
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/backend',
        env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(f'[uvicorn] {line.strip()}')

threading.Thread(target=run_server, daemon=True).start()

print('Waiting for server...')
for i in range(120):
    try:
        if requests.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(f'Server ready ({i+1}s)')
            break
    except: pass
    if i % 10 == 9: print(f'  {i+1}s...')
    time.sleep(1)
else:
    raise RuntimeError('Server failed to start')

tunnel = ngrok.connect(8000)
url = tunnel.public_url
print(f'\n{"="*50}')
print(f'PUBLIC URL: {url}')
print(f'  Docs: {url}/docs')
print(f'  Health: {url}/health')
print(f'{"="*50}')
print(f"\nFor vite.config.js:")
print(f"const BACKEND_URL = '{url}'")

## 10. Test API

In [ ]:
import requests, json, cv2

# Health check
r = requests.get('http://localhost:8000/health')
print(f'Health: {r.status_code}')
print(json.dumps(r.json(), indent=2))

# Detection test
_, enc = cv2.imencode('.jpg', test_image)
r = requests.post(
    'http://localhost:8000/api/detect',
    files={'file': ('test.jpg', enc.tobytes(), 'image/jpeg')},
    data={'save_annotated': 'true'}
)
print(f'\nDetection: {r.status_code}')
d = r.json()
print(f'Persons: {d["stats"]["total_persons"]}, Violations: {d["stats"]["total_violations"]}')

## 11. Download Results

In [ ]:
from google.colab import files as gfiles
for p in ['/content/detection_result.png', '/content/benchmark_yolo.png',
          '/content/backend/uploads/test_annotated.jpg']:
    if os.path.exists(p):
        gfiles.download(p)
        print(f'Downloaded: {os.path.basename(p)}')
print('\nUse these in your thesis!')

---
## Thesis Metrics Summary

| Metric | Value |
|--------|-------|
| YOLO latency (ms) | _fill from Cell 7_ |
| YOLO FPS | _fill from Cell 7_ |
| SAM latency (ms) | _fill from Cell 8_ |
| SAM bypass rate | _fill from Cell 6_ |
| Fast path rate | _fill from Cell 6_ |
| Compliance rate | _fill from Cell 6_ |
| False positives corrected by SAM | _fill from Cell 6_ |

## Test with video

In [ ]:
# ── VIDEO DETECTION WITH TRACKING + SAM 3 ───────────────────────
import sys, os, time, cv2, numpy as np
from google.colab import files
from collections import Counter, defaultdict

sys.path.insert(0, '/content/backend')
os.chdir('/content/backend')

from services.hybrid_detector import get_hybrid_detector
from services.yolo_detector import get_yolo_detector
from services.sam_verifier import get_sam_verifier

# ── CONFIG ──────────────────────────────────────────────────────
PROCESS_EVERY_N = 1     # Track works best with every frame
MAX_FRAMES = 500
MIN_PERSON_ASPECT_RATIO = 1.2  # Filter out buildings (persons are taller than wide)
# ────────────────────────────────────────────────────────────────

# 1. Upload video
# print('Upload a video (MP4/AVI)...')
# uploaded = files.upload()
# video_name = list(uploaded.keys())[0]
# video_path = f'/content/{video_name}'
# with open(video_path, 'wb') as f:
#     f.write(uploaded[video_name])
# video_path = "/content/8965526-uhd_2160_3840_25fps.mp4"
# 2. Open video
cap = cv2.VideoCapture(video_path)
fps_in = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'Video: {video_name} ({w}x{h}, {fps_in:.1f}fps, {total_frames} frames)')

# 3. Setup
out_path = '/content/output_tracked.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_path, fourcc, fps_in, (w, h))

yolo = get_yolo_detector()
yolo.load_model()
detector = get_hybrid_detector()
sam = get_sam_verifier()

from config.settings import settings

# Per-person tracking state
person_status = {}
unique_persons = set()

frame_idx = 0
processed = 0
all_latencies = []
path_counts = Counter()
filtered_as_building = 0

COLORS = {}
def get_color(track_id):
    if track_id not in COLORS:
        np.random.seed(track_id * 7)
        COLORS[track_id] = tuple(int(c) for c in np.random.randint(80, 255, 3))
    return COLORS[track_id]

print('Processing with tracking...')
t_start = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    if MAX_FRAMES and processed >= MAX_FRAMES:
        break

    t0 = time.time()

    # === YOLO with TRACKING ===
    track_results = yolo.model.track(
        frame,
        conf=settings.yolo_confidence_threshold,
        imgsz=settings.yolo_imgsz,
        persist=True,
        tracker='bytetrack.yaml',
        verbose=False
    )

    latency = (time.time() - t0) * 1000
    all_latencies.append(latency)
    processed += 1

    annotated = frame.copy()

    if track_results[0].boxes is not None and track_results[0].boxes.id is not None:
        boxes = track_results[0].boxes.xyxy.cpu().numpy()
        track_ids = track_results[0].boxes.id.cpu().numpy().astype(int)
        cls_ids = track_results[0].boxes.cls.cpu().numpy().astype(int)
        confs = track_results[0].boxes.conf.cpu().numpy()

        for box, tid, cls_id, conf in zip(boxes, track_ids, cls_ids, confs):
            if cls_id != 2:  # Not a person
                continue

            # === FILTER: Remove false persons (buildings, objects) ===
            bw = box[2] - box[0]
            bh = box[3] - box[1]
            aspect_ratio = bh / bw if bw > 0 else 0
            bbox_area = bw * bh
            frame_area = w * h

            if aspect_ratio < 1.2:
                filtered_as_building += 1
                continue
            if bbox_area / frame_area > 0.25:
                filtered_as_building += 1
                continue
            if bh < 80:
                filtered_as_building += 1
                continue
            if conf < 0.45:
                filtered_as_building += 1
                continue

            unique_persons.add(tid)
            bbox = box.tolist()

            # Build person dict
            person = {
                'bbox': bbox,
                'confidence': float(conf),
                'helmet_detected': False,
                'vest_detected': False,
                'no_helmet_detected': False,
                'associated_ppe': []
            }

            # Check PPE associations from same frame
            for b2, c2, conf2 in zip(boxes, cls_ids, confs):
                if c2 == 2:
                    continue
                from utils.bbox_utils import is_inside_bbox
                ppe_bbox = b2.tolist()
                if is_inside_bbox(ppe_bbox, bbox, threshold=0.5):
                    model_names = yolo.model.names
                    name = model_names.get(int(c2), '')
                    if name == 'Helmet':
                        person['helmet_detected'] = True
                    elif name == 'Vest':
                        person['vest_detected'] = True
                    elif name == 'no_helmet':
                        person['no_helmet_detected'] = True

            # Run through 5-path logic
            result, path, used_sam = detector._process_person(person, frame, person_id=tid)
            path_counts[path.value] += 1

            # Update tracking state
            person_status[tid] = {
                'has_helmet': result.has_helmet,
                'has_vest': result.has_vest,
                'decision_path': result.decision_path,
                'is_violation': result.is_violation,
                'sam_activated': result.sam_activated,
                'frames_seen': person_status.get(tid, {}).get('frames_seen', 0) + 1
            }

            # Draw with track ID
            x1, y1, x2, y2 = [int(v) for v in bbox]
            color = (0, 0, 255) if result.is_violation else (0, 255, 0)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

            label = f'ID:{tid} {result.decision_path}'
            if result.sam_activated:
                label += ' [SAM]'
            h_label = 'H:Y' if result.has_helmet else 'H:N'
            v_label = 'V:Y' if result.has_vest else 'V:N'
            label += f' {h_label} {v_label}'

            cv2.putText(annotated, label, (x1, y1-8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 2)

    # Frame overlay
    n_unique = len(unique_persons)
    cv2.putText(annotated, f'Frame:{processed} | {latency:.0f}ms | Unique:{n_unique}',
               (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    out.write(annotated)

    if processed % 30 == 0:
        avg = np.mean(all_latencies[-30:])
        print(f'  Frame {processed}: {latency:.0f}ms (avg {avg:.0f}ms) | Unique persons: {n_unique}')

    if processed == 15:
        cv2.imwrite('/content/sample_tracked.jpg', annotated)

cap.release()
out.release()
total_time = time.time() - t_start

# 5. Results
print(f'\n{"="*50}')
print(f'VIDEO TRACKING RESULTS')
print(f'{"="*50}')
print(f'  Frames processed: {processed}')
print(f'  Total time: {total_time:.1f}s')
print(f'  Avg latency: {np.mean(all_latencies):.1f}ms')
print(f'  Effective FPS: {1000/np.mean(all_latencies):.1f}')
print(f'  UNIQUE persons tracked: {len(unique_persons)}')
print(f'  False persons filtered (buildings): {filtered_as_building}')
print(f'  (vs {sum(path_counts.values())} real person detections)')

print(f'\n  Per-person summary:')
for tid, info in sorted(person_status.items()):
    status = 'VIOLATION' if info['is_violation'] else 'SAFE'
    h = 'Y' if info['has_helmet'] else 'N'
    v = 'Y' if info['has_vest'] else 'N'
    print(f'    Person {tid}: {status} | Helmet:{h} Vest:{v} | Path:{info["decision_path"]} | Seen in {info["frames_seen"]} frames')

print(f'\n  Path distribution:')
total_paths = sum(path_counts.values())
for path, count in path_counts.most_common():
    pct = count / total_paths * 100 if total_paths > 0 else 0
    print(f'    {path}: {count} ({pct:.1f}%)')

fast = path_counts.get('Fast Safe', 0) + path_counts.get('Fast Violation', 0)
if total_paths > 0:
    print(f'\n  SAM bypass rate: {fast/total_paths*100:.1f}%')

# SAM stats
sam_stats = sam.get_stats()
print(f'\n  SAM 3 stats:')
for k, v in sam_stats.items():
    print(f'    {k}: {v:.2f}' if isinstance(v, float) else f'    {k}: {v}')

# Show sample
import matplotlib.pyplot as plt
if os.path.exists('/content/sample_tracked.jpg'):
    img = cv2.imread('/content/sample_tracked.jpg')
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title('Sample Frame with Tracking IDs')
    plt.axis('off')
    plt.show()

# Download
print(f'\nOutput: {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)')
files.download(out_path)


## Trial starts from here

##Cell 1: Setup

In [ ]:
# ── SETUP FOR PIPELINE TEST ──
import sys, os
sys.path.insert(0, '/content/backend')
os.chdir('/content/backend')

# Init database with new table
from database.connection import engine
from database.models import Base
Base.metadata.create_all(bind=engine)

from sqlalchemy import inspect
tables = inspect(engine).get_table_names()
print(f"DB tables: {tables}")
assert 'verified_violations' in tables, "verified_violations table missing!"
print("Setup OK")


##Cell 2: Upload Video + Run Full Pipeline

In [ ]:
# ── FULL SENTRY-JUDGE PIPELINE TEST ──
import sys, os
sys.path.insert(0, '/content/backend')
os.chdir('/content/backend')

from google.colab import files

# Upload video
print('Upload a test video (MP4/AVI)...')
uploaded = files.upload()
video_name = list(uploaded.keys())[0]
video_path = f'/content/{video_name}'
with open(video_path, 'wb') as f:
    f.write(uploaded[video_name])

# Run the pipeline
from run_pipeline import run_pipeline

results = run_pipeline(
    video_path=video_path,
    output_path='/content/output_pipeline.mp4',
    cooldown_seconds=300,
    roi_save_dir='/content/temp_rois',
    camera_zone='CAM-001',
)


##Cell 3: Inspect Results + Download

In [ ]:
# ── INSPECT RESULTS ──
import os, glob
from database.connection import SessionLocal
from database.models import VerifiedViolation

# Query all verified violations
session = SessionLocal()
violations = session.query(VerifiedViolation).all()
print(f"\n{'='*50}")
print(f"VERIFIED VIOLATIONS IN DATABASE: {len(violations)}")
print(f"{'='*50}")

for v in violations:
    print(f"  Person {v.person_id} | {v.violation_type} | {v.timestamp.strftime('%H:%M:%S')} | "
          f"Judge conf: {v.judge_confidence:.2f} | Path: {v.decision_path}")
session.close()

# Show saved ROI images
rois = glob.glob('/content/temp_rois/*.jpg')
print(f"\nSaved ROI evidence images: {len(rois)}")
for r in rois[:5]:
    print(f"  {os.path.basename(r)}")

# Show ROI images visually
import matplotlib.pyplot as plt
import cv2

if rois:
    n = min(6, len(rois))
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, path in zip(axes, rois[:n]):
        img = cv2.imread(path)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(os.path.basename(path)[:30], fontsize=8)
        ax.axis('off')
    plt.suptitle('ROI Evidence Images sent to Judge')
    plt.tight_layout()
    plt.show()

# Download outputs
from google.colab import files as gfiles
for p in ['/content/output_pipeline.mp4']:
    if os.path.exists(p):
        gfiles.download(p)
        print(f"Downloaded: {os.path.basename(p)}")

# Download reports if any
for pdf in glob.glob('/content/backend/reports/*'):
    gfiles.download(pdf)
    print(f"Downloaded report: {os.path.basename(pdf)}")


In [ ]:
# ── TEST JUDGE WITH PERSON VERIFICATION ──────────────────────
import sys, os, glob, cv2, time
import matplotlib.pyplot as plt
sys.path.insert(0, '/content/backend')
os.chdir('/content/backend')

from services.judge import Judge
import queue
import importlib
import services.judge
importlib.reload(services.judge)
from services.judge import Judge
import numpy as np
# Create a dummy queue (we won't use it, just needed for init)
dummy_q = queue.Queue()
judge = Judge(queue=dummy_q, db_session_factory=None, roi_cleanup=False)

print(f"SAM mock mode: {judge.sam.is_mock()}")

# Get all saved ROI images
rois = sorted(glob.glob('/content/temp_rois/*.jpg'))
print(f"Found {len(rois)} ROI images\n")

results = []
for roi_path in rois:
    fname = os.path.basename(roi_path)
    parts = fname.replace('roi_p', '').split('_')
    person_id = parts[0]

    if 'no_helmet' in fname:
        violation_type = 'no_helmet'
    elif 'no_vest' in fname:
        violation_type = 'no_vest'
    else:
        violation_type = 'both_missing'

    roi = cv2.imread(roi_path)
    if roi is None:
        continue

    # Step 0: Person check
    t0 = time.time()
    is_person = judge._is_person(roi)
    person_ms = (time.time() - t0) * 1000

    if not is_person:
        print(f"  Person {person_id} | {violation_type} | NOT A PERSON -> REJECTED ({person_ms:.0f}ms)")
        results.append({
            'person_id': person_id, 'violation_type': violation_type,
            'is_person': False, 'confirmed': False, 'verdict': 'NOT A PERSON',
            'path': roi_path, 'ms': person_ms,
        })
        continue

    # Step 1: PPE check (only if person confirmed)
    t1 = time.time()
    confirmed, confidence = judge._verify_with_sam(roi, violation_type, [])
    total_ms = (time.time() - t0) * 1000

    verdict = "CONFIRMED (real violation)" if confirmed else "REJECTED (PPE found)"
    print(f"  Person {person_id} | {violation_type} | IS person | {verdict} | conf={confidence:.3f} ({total_ms:.0f}ms)")

    results.append({
        'person_id': person_id, 'violation_type': violation_type,
        'is_person': True, 'confirmed': confirmed, 'verdict': verdict,
        'path': roi_path, 'ms': total_ms,
    })

# Visualize
n = min(8, len(results))
if n > 0:
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    if n == 1: axes = np.array([[axes]])
    axes = np.array(axes).reshape(rows, cols) if n > 1 else axes

    import numpy as np
    for i, r in enumerate(results[:n]):
        row, col = i // cols, i % cols
        img = cv2.imread(r['path'])
        axes[row][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

        if not r['is_person']:
            color, border = 'blue', 'blue'
            title = f"P{r['person_id']}\nNOT A PERSON\nREJECTED"
        elif r['confirmed']:
            color, border = 'red', 'red'
            title = f"P{r['person_id']} - {r['violation_type']}\nCONFIRMED VIOLATION"
        else:
            color, border = 'green', 'green'
            title = f"P{r['person_id']} - {r['violation_type']}\nREJECTED (PPE found)"

        axes[row][col].set_title(title, fontsize=9, color=color, fontweight='bold')
        axes[row][col].axis('off')

    for i in range(n, rows * cols):
        axes[i // cols][i % cols].axis('off')

    plt.suptitle('JUDGE 2-STEP VERIFICATION', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/judge_2step_verdicts.png', dpi=150)
    plt.show()

# Summary
not_person = sum(1 for r in results if not r['is_person'])
confirmed = sum(1 for r in results if r['is_person'] and r['confirmed'])
rejected = sum(1 for r in results if r['is_person'] and not r['confirmed'])

print(f"\n{'='*50}")
print(f"JUDGE 2-STEP SUMMARY")
print(f"{'='*50}")
print(f"  Total ROIs: {len(results)}")
print(f"  Step 0 - NOT a person: {not_person} (buildings/objects filtered!)")
print(f"  Step 1 - CONFIRMED violations: {confirmed}")
print(f"  Step 1 - REJECTED (PPE found): {rejected}")


In [ ]:
from ultralytics import YOLO

# Load your specific model (update the path if needed)
model = YOLO("/content/drive/MyDrive/ppe_models/best.pt") # Or your Kaggle path

# Print the dictionary of classes
print("Classes baked into this model:")
print(model.names)